# Uncertainty Estimation

Compare MC Dropout and deep ensembles on heteroscedastic regression.

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split

from prob_ml.data.regression_dgp import RegressionDGPConfig, generate_regression_data
from prob_ml.models.trainer import TrainConfig
from prob_ml.uncertainty.mc_dropout import mc_dropout_predict
from prob_ml.uncertainty.deep_ensemble import fit_deep_ensemble
from prob_ml.uncertainty.metrics import gaussian_interval, picp, rmse

In [ ]:
data = generate_regression_data(RegressionDGPConfig(n_samples=3000, seed=42))
X_train, X_test, y_train, y_test = train_test_split(data.X, data.y, test_size=0.3, random_state=42)

mc = mc_dropout_predict(X_train, y_train, X_test, n_mc_samples=30, config=TrainConfig(epochs=40, dropout=0.2, seed=42))
ens = fit_deep_ensemble(X_train, y_train, X_test, n_members=5, config=TrainConfig(epochs=40, seed=42), seed=42)

for name, res in [('MC Dropout', mc), ('Deep Ensemble', ens)]:
    lo, hi = gaussian_interval(res.mean, res.std, alpha=0.1)
    print(f"{name}: RMSE={rmse(res.mean, y_test):.3f}, PICP={picp(y_test, lo, hi):.3f}")

In [ ]:
order = np.argsort(mc.std)
plt.scatter(mc.std, np.abs(y_test - mc.mean), alpha=0.4, s=10, label='MC Dropout')
plt.scatter(ens.std, np.abs(y_test - ens.mean), alpha=0.4, s=10, label='Deep Ensemble')
plt.xlabel('Predicted std')
plt.ylabel('Absolute error')
plt.legend()
plt.title('Uncertainty vs error')
plt.show()